In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans


df = pd.read_csv('data/turistik_xizmatlar.csv')
print(df.head())

   xizmat_turi yonalish mavsum  davomiylik_kun  narx_usd  reyting  \
0    Tur paket    Nukus    Yoz            14.0    1035.0      4.1   
1  Gid xizmati  Chimyon    Kuz             3.0     273.0      3.8   
2  Gid xizmati     Xiva    Kuz             7.0     381.0      4.0   
3    Transport    Nukus    Yoz            10.0     720.0      3.8   
4    Transport     Xiva   Qish             7.0     363.0      4.2   

   guruh_hajmi   talab  
0            2    Past  
1            3  Yuqori  
2            9    Past  
3           10    Past  
4            4  Yuqori  


In [ ]:

np.random.seed(42)
n = 60
kun = np.random.randint(1, 15, n)
narx = kun * np.random.randint(40, 90, n) + np.random.randint(50, 200, n)
reyting = np.round(np.random.uniform(3.5, 5.0, n), 1)
guruh = np.random.randint(2, 20, n)
talab = np.where((reyting > 4.2) & (narx < 700), "Yuqori", "Past")
 
df = pd.DataFrame({"davomiylik": kun, "narx": narx, "reyting": reyting,
                   "guruh": guruh, "talab": talab})
df.loc[[3, 10], "reyting"] = np.nan
df = pd.concat([df, df.iloc[[0, 1]]])
df.to_csv("turistik_xizmatlar.csv", index=False)
 

In [ ]:
df = pd.read_csv("turistik_xizmatlar.csv")
print(df.head(10))
print("Bo'sh qiymatlar:\n", df.isnull().sum())
print("Takrorlar:", df.duplicated().sum())
 
df = df.drop_duplicates()
df["reyting"] = df["reyting"].fillna(df["reyting"].median())

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(12, 9))
 
ax[0, 0].hist(df["narx"], bins=10)
ax[0, 0].set_title("Histogram: narx")
 
df["talab"].value_counts().plot(kind="bar", ax=ax[0, 1])
ax[0, 1].set_title("Bar Chart: talab")
 
ax[1, 0].scatter(df["davomiylik"], df["narx"])
ax[1, 0].set_title("Scatter Plot: davomiylik va narx")
 
sns.heatmap(df[["davomiylik", "narx", "reyting", "guruh"]].corr(),
            annot=True, ax=ax[1, 1])
ax[1, 1].set_title("Heatmap: korrelyatsiya")
 
plt.tight_layout()
plt.show()

In [ ]:
X = df[["davomiylik", "narx", "reyting", "guruh"]]
y = df["talab"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
acc_lr = accuracy_score(y_test, lr.predict(X_test))
print("Logistic Regression accuracy:", acc_lr)

In [ ]:

dt = DecisionTreeClassifier(max_depth=3, random_state=42)
dt.fit(X_train, y_train)
acc_dt = accuracy_score(y_test, dt.predict(X_test))
print("Decision Tree accuracy:", acc_dt)

In [ ]:
data = df[["narx", "davomiylik"]]
scaled = StandardScaler().fit_transform(data)
df["klaster"] = KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(scaled)

In [ ]:

plt.scatter(df["davomiylik"], df["narx"], c=df["klaster"])
plt.xlabel("Davomiylik (kun)")
plt.ylabel("Narx (USD)")
plt.title("K-Means klasterlari")
plt.show()
 
print("Obyektlar soni:\n", df["klaster"].value_counts().sort_index())
print("O'rtacha qiymatlar:\n", df.groupby("klaster")[["narx", "davomiylik"]].mean().round(1))
 
print(f"Logistic Regression: {acc_lr:.2f}, Decision Tree: {acc_dt:.2f}")